# Chain-of-Thought Prompting (CoT)


Instrui o modelo a externalizar **passos de raciocínio intermediários** antes da resposta final. Wei et al. (2022) demonstraram que essa simples mudança elicita capacidades emergentes em modelos com >100B parâmetros, praticamente ausentes em menores. Self-Consistency (Wang et al., 2022) amplifica a robustez: N amostras com temperatura > 0 + voto majoritário superam consistentemente o greedy decoding em benchmarks de aritmética e lógica.

**Referências:** Wei et al. (2022) arXiv:2201.11903 · Wang et al. (2022) arXiv:2203.11171

In [ ]:
!pip install -q --upgrade langchain-ollama langchain-core python-dotenv langchain requests


In [3]:
from pathlib import Path
import os, json, re, unicodedata
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama


In [4]:
from pathlib import Path
import os, json, re, unicodedata
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# ── Configuração local Ollama ──────────────────────────────────────────────
load_dotenv(override=True)
MODEL_NAME = "gemma3:4b"
CREATIVE_MODEL_NAME = "phi4-mini"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")


# ── Constants ──────────────────────────────────────────────────────────────

# env_candidates = [Path("code/.env"), Path(".env")]
# env_path = next((p for p in env_candidates if p.exists()), None)
# if env_path is None:
#     env_path = Path(".env")

# load_dotenv(dotenv_path=env_path, override=True)

# ── Clientes LLM ───────────────────────────────────────────────────────────
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)
llm_criativo = ChatOllama(
    model=CREATIVE_MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7,
)

print(f"✓ Ollama | modelo padrão: {MODEL_NAME}")
print(f"✓ Ollama | modelo criativo: {CREATIVE_MODEL_NAME}")
print(f"✓ Ollama | base_url: {OLLAMA_BASE_URL}")
print("Configuração concluída.")

# ── Funções auxiliares ────────────────────────────────────────────────────────
def normalizar(texto: str) -> str:
    """Lowercase + strip diacritics for keyword matching."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))

def extrair_json(texto: str) -> dict:
    """Strip markdown fences and parse the first JSON object found."""
    texto = texto.strip()
    if texto.startswith("```"):
        texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.S).strip()
    inicio = texto.find("{")
    fim    = texto.rfind("}")
    if inicio != -1 and fim != -1 and fim > inicio:
        texto = texto[inicio : fim + 1]
    return json.loads(texto)

def chamar_texto(llm_client, prompt_template, alternativa: str, **kwargs) -> str:
    """Call the LLM and return a string; fall back gracefully."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return getattr(bruto, "content", str(bruto)).strip()
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa

def pedir_json(llm_client, prompt_template, alternativa: dict, **kwargs) -> dict:
    """Call the LLM and parse JSON; fall back gracefully."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return extrair_json(getattr(bruto, "content", str(bruto)))
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa



✓ Ollama | modelo padrão: gemma3:4b
✓ Ollama | modelo criativo: phi4-mini
✓ Ollama | base_url: http://172.18.224.1:11434
Configuração concluída.


## 01. Zero-shot CoT

Adicione *"Vamos pensar passo a passo."* — sem exemplos, o modelo raciocina por conta própria.

In [5]:
# ── Zero-shot CoT ──────────────────────────────────────────────────────────
# Adicionar "Vamos pensar passo a passo." já é suficiente para induzir raciocínio.
# Nenhum exemplo é fornecido — o modelo depende apenas do conhecimento pré-treinado.

PERGUNTA_BASICA = (
    "Um cliente comprou 3 cadernos a R$12 cada e pagou com uma nota de R$50. "
    "Qual é o troco?"
)

prompt_zero_shot_cot = PromptTemplate(
    input_variables=["pergunta"],
    template=(
        "Vamos pensar passo a passo.\n"
        "Mostre cada etapa de cálculo antes de dar a resposta final.\n\n"
        "Pergunta: {pergunta}\n"
        "Resposta:"
    ),
)

resposta_zero_shot = chamar_texto(
    llm,
    prompt_zero_shot_cot,
    alternativa=(
        "Passo 1: custo total = 3 × R$12 = R$36.\n"
        "Passo 2: troco = R$50 − R$36 = R$14.\n"
        "Resposta: o cliente recebe R$14,00 de troco."
    ),
    pergunta=PERGUNTA_BASICA,
)

print("Pergunta:")
print(PERGUNTA_BASICA)
print("\nResposta (Zero-shot CoT):")
print(resposta_zero_shot)

Pergunta:
Um cliente comprou 3 cadernos a R$12 cada e pagou com uma nota de R$50. Qual é o troco?

Resposta (Zero-shot CoT):
Aqui está o cálculo passo a passo:

1. **Calcular o custo total dos cadernos:** 3 cadernos * R$12/caderno = R$36
2. **Calcular o troco:** R$50 (valor pago) - R$36 (custo dos cadernos) = R$14

Resposta: O troco é R$14.


## 02. Few-shot CoT

Forneça 2–3 exemplos resolvidos. O modelo imita o padrão de raciocínio que você demonstrou.

In [7]:
# ── Few-shot CoT ──────────────────────────────────────────────────────────
# Fornecemos exemplos resolvidos — o modelo imita o raciocínio explícito.
# Útil quando você quer controlar o estilo e a granularidade do passo a passo.

PERGUNTA_LIVRARIA = "Uma livraria vendeu 4 livros a R$35 cada e 2 livros a 50. Quanto arrecadou no total?"

EXEMPLOS_COT = (
    "P: João tinha 5 maçãs e deu 2 para Ana. Quantas ficou?\n"
    "R: Começa com 5. Retira 2. Resultado = 5 − 2 = 3.\n\n"
    "P: Um caderno custa R$12 e uma caneta custa R$8. Qual o total?\n"
    "R: Soma os dois. R$12 + R$8 = R$20.\n\n"
)

prompt_few_shot_cot = PromptTemplate(
    input_variables=["exemplos", "pergunta"],
    template=(
        "Resolva no mesmo estilo dos exemplos — raciocínio curto, resultado claro.\n\n"
        "{exemplos}"
        "P: {pergunta}\n"
        "R:"
    ),
)

resposta_few_shot = chamar_texto(
    llm,
    prompt_few_shot_cot,
    alternativa="4 livros × R$35 = R$140. Arrecadação total: R$140.",
    exemplos=EXEMPLOS_COT,
    pergunta=PERGUNTA_LIVRARIA,
)

print("Pergunta:")
print(PERGUNTA_LIVRARIA)
print("\nResposta (Few-shot CoT):")
print(resposta_few_shot)

Pergunta:
Uma livraria vendeu 4 livros a R$35 cada e 2 livros a 50. Quanto arrecadou no total?

Resposta (Few-shot CoT):
R: Calcula o valor dos 4 livros: 4 x R$35 = R$140. Calcula o valor dos 2 livros: 2 x R$50 = R$100. Soma os dois valores: R$140 + R$100 = R$240.


## 03. Auto-CoT

Gere exemplos automaticamente a partir de uma pergunta-guia e reutilize-os como few-shot.

In [8]:
# ── Auto-CoT ──────────────────────────────────────────────────────────────
# Gera automaticamente um exemplo resolvido para um tipo de problema
# e usa esse exemplo como guia para resolver uma nova pergunta.
# Útil quando você tem muitas perguntas similares e não quer anotar exemplos à mão.

def gerar_exemplo_cot(pergunta_guia: str) -> str:
    """Ask the LLM to produce one solved Q/A pair for a given problem type."""
    prompt = PromptTemplate(
        input_variables=["pergunta_guia"],
        template=(
            "Gere um exemplo resolvido no formato P/R para este tipo de problema.\n"
            "Mostre o raciocínio passo a passo de forma curta.\n\n"
            "Tipo de problema: {pergunta_guia}\n"
            "Exemplo:"
        ),
    )
    return chamar_texto(
        llm,
        prompt,
        alternativa=(
            "P: Se 3 ingressos custam R$90, quanto custa 1?\n"
            "R: R$90 ÷ 3 = R$30 por ingresso."
        ),
        pergunta_guia=pergunta_guia,
    )


def resolver_com_exemplo_cot(exemplo: str, pergunta: str) -> str:
    """Solve a new question using the auto-generated example as a few-shot."""
    prompt = PromptTemplate(
        input_variables=["exemplo", "pergunta"],
        template=(
            "Use o exemplo abaixo como guia para resolver a pergunta.\n\n"
            "Exemplo:\n{exemplo}\n\n"
            "Pergunta: {pergunta}\n"
            "Resposta:"
        ),
    )
    return chamar_texto(
        llm,
        prompt,
        alternativa="R$180 ÷ 6 = R$30 por ingresso.",
        exemplo=exemplo,
        pergunta=pergunta,
    )


PERGUNTA_GUIA  = "Calcular o preço unitário a partir do total e da quantidade."
PERGUNTA_NOVA  = "Se 6 ingressos custam R$180, quanto custa 1 ingresso?"

exemplo_gerado = gerar_exemplo_cot(PERGUNTA_GUIA)
resposta_auto  = resolver_com_exemplo_cot(exemplo_gerado, PERGUNTA_NOVA)

print("Exemplo gerado automaticamente:")
print(exemplo_gerado)
print("\nPergunta nova:")
print(PERGUNTA_NOVA)
print("\nResposta (Auto-CoT):")
print(resposta_auto)

Exemplo gerado automaticamente:
## Exemplo Resolvido: Cálculo de Preço Unitário (Formato P/R)

**Problema:** Uma loja vende camisetas por R$ 35,00. Se o total da compra foi de R$ 140,00, qual o número de camisetas compradas?

**P (Problema):** Calcular a quantidade de camisetas compradas com base no total da compra e no preço unitário da camiseta.

**R (Resolução):**

1. **Definir Variáveis:**
   *  `x` = número de camisetas compradas

2. **Montar a Equação:**
   *  O preço total da compra é calculado multiplicando o preço unitário pelo número de camisetas:  `35 * x = 140`

3. **Resolver a Equação:**
   *  Dividir ambos os lados da equação por 35: `x = 140 / 35`
   *  `x = 4`

4. **Verificar a Resposta:**
   *  4 camisetas * R$ 35,00/camiseta = R$ 140,00 (Total da compra correto)

**Resposta:** Foram compradas 4 camisetas.

**Formato P/R:**

*   **P (Problema):** Calcular a quantidade de camisetas compradas com base no total da compra e no preço unitário da camiseta.
*   **R (Resolução

## 04. Self-Consistency

Gere múltiplas respostas com temperatura > 0 e eleja a mais frequente por votação majoritária.

In [9]:
# ── Self-Consistency ──────────────────────────────────────────────────────
# Gera N respostas independentes (temperatura > 0) e seleciona a mais frequente.
# Reduz a variância em perguntas onde o modelo pode divergir entre amostras.
from collections import Counter

N_AMOSTRAS = 5
PERGUNTA_MAÇAS = (
    "Uma caixa tem 12 maçãs. Saem 5 e depois entram mais 8. "
    "Quantas maçãs ficam na caixa?"
)

def extrair_numero_final(texto: str) -> str:
    """Return the last number found in the text (best-effort extraction)."""
    numeros = re.findall(r"-?\d+(?:[.,]\d+)?", texto)
    return numeros[-1] if numeros else texto.strip()


def amostrar_resposta_cot(pergunta: str) -> str:
    prompt = PromptTemplate(
        input_variables=["pergunta"],
        template=(
            "Resolva passo a passo e termine respondendo apenas com o número final.\n\n"
            "Pergunta: {pergunta}\n"
            "Resposta:"
        ),
    )
    return chamar_texto(
        llm_criativo,
        prompt,
        alternativa="12 − 5 + 8 = 15",
        pergunta=pergunta,
    )


amostras       = [amostrar_resposta_cot(PERGUNTA_MAÇAS) for _ in range(N_AMOSTRAS)]
votos          = [extrair_numero_final(s) for s in amostras]
resposta_final = Counter(votos).most_common(1)[0][0]

print("Pergunta:")
print(PERGUNTA_MAÇAS)
print(f"\nAmostras geradas (N={N_AMOSTRAS}):")
for i, s in enumerate(amostras, 1):
    print(f"  {i}. {s}")
print(f"\nVoto majoritário → {resposta_final}")

Pergunta:
Uma caixa tem 12 maçãs. Saem 5 e depois entram mais 8. Quantas maçãs ficam na caixa?

Amostras geradas (N=5):
  1. 1. Comece com 12 maçãs.
2. Subtraia as 5 maçãs saídas, resultando em 7 maçãs (12 - 5 = 7).
3. Adicione os 8 novas maçãs, ficando com um total de 15 maçãs na caixa (7 + 8 = 15).

Resposta final: 15
  2. 1 debegin with 12 maçãs.
2 saem 5, então temos 12 - 5 = 7 maçãs restantes.
3 entrem mais 8, então agora temos 7 + 8 = 15 maçãs.

A resposta final é 15.
  3. 1. Comece com 12 maçãs.
2. Saem 5 maçãs, então resta 12 - 5 = 7 maçãs.
3. Entram em mais 8 maçãs, tornando o total de 7 + 8 = 15 maçãs.

O número final de maçãs na caixa é: **15**
  4. Passo 1: Comece com 12 maçãs.
Passo 2: Saíam 5 maçãs, então você subtrai 5 de 12, o que deixa 7 maçãs.
Passo 3: Entrarão mais 8 maçãs na caixa, então adiciona 8 a 7, resultando em 15.

Resposta final: 15.
  5. Passo 1: Comece com o número inicial de maçãs, que é 12.
Passo 2: Subtraia as 5 maçãs saídas da caixa (12 - 5 = 7).
Passo